In [ ]:
from pathlib import Path

import io
import onnx
import onnxruntime.training.onnxblock as onnxblock
import torch
# from onnxsim import simplify

from model import Model

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# Create a classifier instance
device = "cpu"
batch_size = 64
num_classes = 10
channels = 3
img_h, img_w = 128, 128
frozen_layer_num = 10

model_name = 'resnet18' # resnet18, resnet34, mobilenetv2_100, efficientnet_b2, visformer_tiny

# artifacts path
artifacts_path = Path(f'./artifacts/{model_name}_frozen_layer_{frozen_layer_num}/cpu')
artifacts_path.mkdir(parents=True, exist_ok=True)

pt_model = Model(model_name=model_name, num_classes=num_classes, channels=channels, frozen_layer_num=frozen_layer_num, img_size=img_h)
print(f'Model trainable parameters: {count_parameters(pt_model)}')

for param in pt_model.named_parameters():
    if 'frozen_features' in param[0] or 'bn' in param[0] or 'downsample.1' in param[0]:
        param[1].requires_grad = False
    else:
        param[1].requires_grad = True

torch.save(pt_model, artifacts_path / f'{model_name}.pth')
pt_model = torch.load(artifacts_path / f'{model_name}.pth')
for param in pt_model.named_parameters():
    if param[1].requires_grad:
        print(param[0].ljust(40), '->\t', param[1].requires_grad)

print(f'Model trainable parameters: {count_parameters(pt_model)}')

### Generating forward-only graph

In [ ]:
# Generate a random input.
model_inputs = (torch.randn(batch_size, channels, img_h, img_w, device=device),)

model_outputs = pt_model(*model_inputs)
if isinstance(model_outputs, torch.Tensor):
    model_outputs = [model_outputs]
    
input_names = ["input"]
output_names = ["output"]
dynamic_axes = {"input": {0: "batch_size"}, "output": {0: "batch_size"}}

f = io.BytesIO()
torch.onnx.export(
    pt_model,
    model_inputs,
    f,
    input_names=input_names,
    output_names=output_names,
    opset_version=17,
    do_constant_folding=False,
    training=torch.onnx.TrainingMode.TRAINING,
    dynamic_axes=dynamic_axes,
    export_params=True,
    keep_initializers_as_inputs=False,
)

onnx_model = onnx.load_model_from_string(f.getvalue())
onnx.save(onnx_model, artifacts_path / f'{model_name}.onnx')

### Generating training graph

Generating artifacts based on forward-only graph and the specified loss function using onnxblock library

In [ ]:
# Creating a class with a Loss function
class CustomTrainingBlock(onnxblock.TrainingBlock):
    def __init__(self):
        super(CustomTrainingBlock, self).__init__()
        self.loss = onnxblock.loss.CrossEntropyLoss()

    def build(self, output_name):
        return self.loss(output_name), output_name

In [ ]:
# Build the onnx model with loss
training_block = CustomTrainingBlock()
for param in onnx_model.graph.initializer:   
    if 'frozen_features' in param.name or 'bn' in param.name or 'downsample.1' in param.name:
        training_block.requires_grad(param.name, False)
    else:
        print(param.name.ljust(40), '\t--->\tlearnable')
        training_block.requires_grad(param.name, True)

# Building training graph and eval graph
model_params = None
with onnxblock.base(onnx_model):
    build_output = training_block(*[output.name for output in onnx_model.graph.output])
    print('Graph output nodes:', build_output)
    training_model, eval_model = training_block.to_model_proto()
    model_params = training_block.parameters()

# Building the optimizer graph
optimizer_block = onnxblock.optim.AdamW()
with onnxblock.empty_base() as accessor:
    _ = optimizer_block(model_params)
    optimizer_model = optimizer_block.to_model_proto()

In [ ]:
# Saving all the files to use them later for the training.
onnxblock.save_checkpoint(training_block.parameters(), artifacts_path / 'checkpoint')

ir_version = 9
opset_import_version = 17

training_model.ir_version = ir_version
# print('Simplifying ONNX training model...')
# training_model, check = simplify(training_model)
# assert check, "Simplified ONNX training model could not be validated"
onnx.save(training_model, artifacts_path / 'training_model.onnx')

optimizer_model.ir_version = ir_version
optimizer_model.opset_import[1].version = opset_import_version
onnx.save(optimizer_model, artifacts_path / 'optimizer_model.onnx')

eval_model.ir_version = ir_version
# print('Simplifying ONNX eval model...')
# eval_model, check = simplify(eval_model)
# assert check, "Simplified ONNX eval model could not be validated"
onnx.save(eval_model, artifacts_path / 'eval_model.onnx')

print(f'Artifacts saved in {artifacts_path} directory')